# Matminer 结构特征提取器学习

In [2]:
import pandas as pd
import numpy as np
from pymatgen.core import Structure

In [3]:
import os
from mp_api.client import MPRester
import pandas as pd
# Set the API key for Materials Project
API_KEY = os.getenv("MP_API_KEY")
need_fields = ['material_id', 'formula_pretty', 'composition', 'band_gap', 'is_gap_direct',
                'formation_energy_per_atom', 'energy_above_hull', 'volume', 'density', 'density_atomic',
                'symmetry', 'nsites', 'structure']

In [4]:
with MPRester(API_KEY) as mpr:
    # Get the data for the specified fields
    docs = mpr.materials.summary.search(
        fields = need_fields,
        material_ids=["mp-1204356", "mp-22862", "mp-3953"]
    )

Retrieving SummaryDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
SiC = docs[0]
CaCO3 = docs[1]
NaCl = docs[2]

In [6]:
# Convert the data to a pandas DataFrame
dict = {
    'material_id': [SiC.material_id, NaCl.material_id, CaCO3.material_id],
    'formula_pretty': [SiC.formula_pretty, NaCl.formula_pretty, CaCO3.formula_pretty],
    'composition': [SiC.composition, NaCl.composition, CaCO3.composition],
    'symmetry': [SiC.symmetry, NaCl.symmetry, CaCO3.symmetry],
    'nsites': [SiC.nsites, NaCl.nsites, CaCO3.nsites],
    'structure': [SiC.structure, NaCl.structure, CaCO3.structure]
}
df_raw = pd.DataFrame(dict)
df_raw

,material_id,formula_pretty,composition,symmetry,nsites,structure
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]"
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0...."
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...


In [59]:
# 添加氧化态信息
# CrystalNN 算法在判断原子间键合时依赖于原子的氧化态信息来获得最佳结果。
from pymatgen.analysis.bond_valence import BVAnalyzer
bv = BVAnalyzer()
df_raw['oxi_structure'] = None
for index, row in df_raw.iterrows():
    structure = row['structure']
    bv_analyzer = BVAnalyzer()
    oxidation_states = bv_analyzer.get_oxi_state_decorated_structure(structure)
    df_raw.at[index, 'oxi_structure'] = oxidation_states

In [60]:
df_raw

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ..."
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ..."
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...


In [61]:
structure = df_raw['oxi_structure'][0]
structure

Structure Summary
Lattice
    abc : 3.951402922131455 3.951402172480169 3.951402
 angles : 59.99998470065274 59.99999097646918 59.99999737523848
 volume : 43.625325465502776
      A : np.float64(3.422015) np.float64(0.0) np.float64(1.975702)
      B : np.float64(1.140671) np.float64(3.226306) np.float64(1.975702)
      C : np.float64(0.0) np.float64(0.0) np.float64(3.951402)
    pbc : True True True
PeriodicSite: Na (Na+) (0.0, 0.0, 0.0) [0.0, 0.0, 0.0]
PeriodicSite: Cl (Cl-) (2.281, 1.613, 3.951) [0.5, 0.5, 0.5]

## Bonding

Structure featurizers based on bonding.

### 1. BondFractions

- Matminer 库中用于计算晶体结构中各种化学键比例的特征提取器。

- BondFractions is based on the "sum over bonds" in the **Bag of Bonds approach**,
based on a method by **Hansen et. al** "Machine Learning Predictions of Molecular
Properties: Accurate Many-Body Potentials and Nonlocality in Chemical Space"(2015).

In [105]:
from matminer.featurizers.structure.bonding import BondFractions

In [106]:
df = df_raw.copy()

In [107]:
from pymatgen.analysis.local_env import VoronoiNN, JmolNN, MinimumDistanceNN
# 1. 初始化特征提取器
bf = BondFractions()
# 2. 拟合特征提取器(确定要分析的键类型)
bf.fit(df['oxi_structure'])

BondFractions()

In [108]:
# 3. 特征化结构(提取键比例)
df[bf.feature_labels()] = None
for index, row in df.iterrows():
    structure = row['oxi_structure']
    features = bf.featurize(structure)
    # 将特征添加到 DataFrame 中
    for i, label in enumerate(bf.feature_labels()):
        df.at[index, label] = features[i]
df

e:\software2\Anaconda\Lib\site-packages\pymatgen\analysis\local_env.py:3933: UserWarning: CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.
  nn_data = self.get_nn_data(structure, n)


,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,C4+ - C4+ bond frac.,C4+ - Ca2+ bond frac.,C4+ - O2- bond frac.,C4- - C4- bond frac.,C4- - Si4+ bond frac.,Ca2+ - Ca2+ bond frac.,Ca2+ - O2- bond frac.,Cl- - Cl- bond frac.,Cl- - Na+ bond frac.,Na+ - Na+ bond frac.,O2- - O2- bond frac.,Si4+ - Si4+ bond frac.
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.0,0.0,0.333333,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,0.0,0.0


- 直接对dataframe进行操作

In [68]:
df = df_raw.copy()

In [69]:
# 直接对dataframe进行特征化
bf = BondFractions()
bf.fit_featurize_dataframe(df, "oxi_structure")

BondFractions:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,C4+ - C4+ bond frac.,C4+ - Ca2+ bond frac.,C4+ - O2- bond frac.,C4- - C4- bond frac.,C4- - Si4+ bond frac.,Ca2+ - Ca2+ bond frac.,Ca2+ - O2- bond frac.,Cl- - Cl- bond frac.,Cl- - Na+ bond frac.,Na+ - Na+ bond frac.,O2- - O2- bond frac.,Si4+ - Si4+ bond frac.
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.0,0.0,0.000000,0.0,1.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,1.0,0.0,0.0,0.0
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.0,0.0,0.333333,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,0.0,0.0


### 2. BagofBands

Matminer 中的一个强大特征提取器，基于 Hansen 等人在 2015 年提出的方法，用于创建固定长度的结构表示向量。它通过对库仑矩阵的处理，表示晶体结构中各种原子对之间的相互作用。

In [109]:
from matminer.featurizers.structure.bonding import BagofBonds
df = df_raw.copy()

In [72]:
bob = BagofBonds()
bob.fit([NaCl.structure])

BagofBonds()

In [75]:
features = bob.featurize(NaCl.structure)
labels = bob.feature_labels()
feat_df = pd.DataFrame([features], columns=labels)
feat_df

,Na site #0,Cl site #0,Na - Cl bond #0,Na - Cl bond #1
0,157.874667,448.794386,10.223729,10.223729


In [77]:
# 直接对dataframe进行特征化
bob = BagofBonds()
df = bob.fit_featurize_dataframe(df, "oxi_structure")
df

BagofBonds:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,Na+ site #0,Cl- site #0,Na+ - Cl- bond #0,...,Si4+ - C4- bond #712,Si4+ - C4- bond #713,Si4+ - C4- bond #714,Si4+ - C4- bond #715,Si4+ - C4- bond #716,Si4+ - C4- bond #717,Si4+ - C4- bond #718,Si4+ - C4- bond #719,Si4+ - C4- bond #720,Si4+ - C4- bond #721
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.000000,0.000000,0.000000,...,60.728955,60.728955,60.734135,60.734135,60.743214,60.743214,60.749668,60.749668,60.777604,60.777604
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",157.874667,448.794386,10.223729,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


理解结果:
BagofBonds 生成的特征向量由多个"袋子"（bags）组成，每个袋子对应一种元素对（键）或单个元素。特征标签的形式为：

- 对于键: Element1 - Element2 bond #N
- 对于单个元素: Element site #N
其中 N 是该键或元素在袋子中的序号。

### 3. GlobalInstabilityIndex

Matminer 中用于评估晶体结构稳定性的特征提取器。它通过计算结构中所有原子的键价和(BVS)与其理论氧化态的偏差来评估结构的稳定性。

**注意事项:**
- 必须提供**氧化态**：结构中每个原子都必须有氧化态信息，否则计算会失败

- 仅适用于离子化合物：计算依赖于阴离子和阳离子之间的键价，所以主要适用于氧化物、氟化物等离子化合物

- 需要有表格参数：用于无序结构时，需要设置

##### 利用全局不稳定指数(GII)判断晶体结构稳定性

GII是评估晶体结构稳定性的重要指标，特别适用于离子化合物。它量化了原子实际键价和与理论氧化态之间的偏差。

**标准判断阈值**

| GII 值范围 | 稳定性评估 | 解释 |
|------------|------------|------|
| < 0.05     | 极其稳定   | 几乎无键价应力，理想结构 |
| 0.05-0.10  | 非常稳定   | 轻微键价应力，常见于实验验证的稳定结构 |
| 0.10-0.20  | 中等稳定   | 存在一定键价应力，结构可能有轻微畸变 |
| 0.20-0.30  | 轻度不稳定 | 明显键价应力，可能需要结构松弛或高温相 |
| 0.30-0.60  | 中度不稳定 | 高键价应力，结构可能只在特定条件下稳定 |
| > 0.60     | 高度不稳定 | 极高键价应力，结构可能不可合成或参数不适用 |

**材料类型与GII值解释**

不同类型的材料体系对GII值的容忍度不同：

- **简单离子晶体**（如NaCl）：通常GII < 0.10
- **氧化物钙钛矿**：通常GII < 0.20
- **复杂氧化物**：可接受的GII范围可达0.30
- **玻璃态材料和非晶材料**：GII概念应用受限

**GII指标的局限性**

1. **依赖精确的氧化态**：如氧化态分配不准确，GII值可能误导
2. **需要完整的键价参数**：一些罕见元素组合的键价参数可能不可用
3. **不考虑温度效应**：高温可能稳定高GII结构
4. **有限的化学敏感性**：无法完全捕捉共价和金属键效应

**全面评估结构稳定性**

建议将GII与其他指标结合使用：
- 结构能量计算（DFT）
- 声子谱分析
- 热力学稳定性（形成能）
- 化学键分析


In [110]:
from matminer.featurizers.structure.bonding import GlobalInstabilityIndex
df = df_raw.copy()

In [80]:
# 1. 加载包含氧化态的结构
# 注意：结构必须包含氧化态信息，否则GII计算会失败
bv = BVAnalyzer()
structure = bv.get_oxi_state_decorated_structure(NaCl.structure)
# 2. 初始化特征提取器
gii_featurizer = GlobalInstabilityIndex(r_cut=4.0)  # r_cut是搜索相邻原子的距离
# 3. 检查结构是否满足预检条件
if gii_featurizer.precheck(structure):
    # 4. 提取特征
    gii_value = gii_featurizer.featurize(structure)[0]
    print(f"全局不稳定指数 (GII): {gii_value:.4f}")
    
    # 判断结构稳定性
    if gii_value < 0.1:
        print("结构非常稳定")
    elif gii_value < 0.2:
        print("结构有一定应力")
    else:
        print("结构可能不稳定")
else:
    print("结构不满足GII计算条件")

e:\software2\Anaconda\Lib\site-packages\matminer\utils\data.py:703: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  self.params: pd.DataFrame = pd.read_csv(


全局不稳定指数 (GII): 0.0524
结构非常稳定


e:\software2\Anaconda\Lib\site-packages\spglib\spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['equivalent_atoms']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(
e:\software2\Anaconda\Lib\site-packages\spglib\spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['wyckoffs']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(


In [81]:
# 直接对dataframe进行特征化
gii_featurizer = GlobalInstabilityIndex(r_cut=4.0)  # r_cut是搜索相邻原子的距离
df = gii_featurizer.fit_featurize_dataframe(df, "oxi_structure")
df

e:\software2\Anaconda\Lib\site-packages\matminer\utils\data.py:703: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  self.params: pd.DataFrame = pd.read_csv(


GlobalInstabilityIndex:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,global instability index
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.092213
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.052373
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.092524


### 4. StructureHeterogeneity

Matminer 中的一个特征提取器，用于计算晶体结构中键长和原子体积的变化程度。这个类通过 Voronoi 镶嵌分析，提取描述结构不均匀性的特征，这些特征对预测材料性质（如形成能）非常有用

In [111]:
from matminer.featurizers.structure.bonding import StructuralHeterogeneity
bv = BVAnalyzer()
structure = bv.get_oxi_state_decorated_structure(NaCl.structure)

In [92]:
# 初始化特征提取器
sh = StructuralHeterogeneity(
    weight="area",  # 可选: "area"(默认), "solid_angle", "volume"
    stats=("minimum", "maximum", "mean", "range", "avg_dev")  # 自定义统计量
)
# 提取特征
features = sh.featurize(structure)

# 获取特征标签
feature_labels = sh.feature_labels()

# 创建包含特征的数据框
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,mean absolute deviation in relative bond length,max relative bond length,min relative bond length,minimum neighbor distance variation,maximum neighbor distance variation,mean neighbor distance variation,range neighbor distance variation,avg_dev neighbor distance variation,mean absolute deviation in relative cell size
0,1.110223e-16,1.0,1.0,2.362613e-07,2.362613e-07,2.362613e-07,1.746508e-16,8.732542e-17,0.0


**返回的特征说明:**

`StructuralHeterogeneity`返回9个特征(使用默认stats参数):

1. **mean absolute deviation in relative bond length**: 所有原子平均键长与整体平均键长偏差
2. **max relative bond length**: 最大相对键长
3. **min relative bond length**: 最小相对键长
4. **minimum neighbor distance variation**: 邻近原子距离变化的最小值
5. **maximum neighbor distance variation**: 邻近原子距离变化的最大值
6. **range neighbor distance variation**: 邻近原子距离变化的范围
7. **mean neighbor distance variation**: 邻近原子距离变化的平均值
8. **avg_dev neighbor distance variation**: 邻近原子距离变化的平均偏差
9. **mean absolute deviation in relative cell size**: 沃罗诺伊单元体积的相对变化

高结构不均匀性的值通常表明材料存在局部结构畸变或多种配位环境，这往往与不寻常的物理或电子性质相关。

这个特征提取器特别适合于研究材料的结构稳定性、形成能和热力学性能。

In [93]:
# 直接对dataframe进行特征化
df = df_raw.copy()
df = sh.fit_featurize_dataframe(df, "oxi_structure", ignore_errors=True)
df

StructuralHeterogeneity:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,mean absolute deviation in relative bond length,max relative bond length,min relative bond length,minimum neighbor distance variation,maximum neighbor distance variation,mean neighbor distance variation,range neighbor distance variation,avg_dev neighbor distance variation,mean absolute deviation in relative cell size
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",1.110223e-16,1.000000,1.0000,2.362613e-07,2.362613e-07,2.362613e-07,1.746508e-16,8.732542e-17,0.000000
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",1.037200e-01,1.075444,0.7407,3.425020e-02,3.984882e-01,2.369495e-01,3.642380e-01,8.107971e-02,0.162066


### 5. MinimumRelativeDistance

是一个强大的结构特征提取器，用于计算晶体结构中每个原子位点到其最近邻的相对距离。不同于使用绝对距离，它采用相对距离计算：f_ij = r_ij / (r^atom_i + r^atom_j)，这考虑了不同元素的原子大小差异。

In [112]:
from matminer.featurizers.structure.bonding import MinimumRelativeDistances

In [96]:
# 初始化特征提取器
mrd = MinimumRelativeDistances(
    cutoff=10.0,            # 搜索近邻的最大距离
    flatten=True,           # 返回固定长度的特征向量
    include_distances=True, # 包含数值距离
    include_species=True    # 包含元素种类信息
)

mrd.fit([structure])

# 提取特征
features = mrd.featurize(structure)

# 获取特征标签
feature_labels = mrd.feature_labels()

# 创建包含特征的数据框
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,site #0 min. rel. dist.,site #0 specie,site #0 neighbor specie(s),site #1 min. rel. dist.,site #1 specie,site #1 neighbor specie(s)
0,0.873145,Na+,"(Na+, Na+)",0.873145,Cl-,"(Cl-, Cl-)"


In [98]:
# 只返回距离值,不包含元素种类信息
mrd_dist_only = MinimumRelativeDistances(include_distances=True, include_species=False)
mrd_dist_only.fit([structure])
# 提取特征
features = mrd_dist_only.featurize(structure)

# 获取特征标签
feature_labels = mrd_dist_only.feature_labels()

# 创建包含特征的数据框
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,site #0 min. rel. dist.,site #1 min. rel. dist.
0,0.873145,0.873145


**输出特征说明**:

当使用默认参数时(flatten=True, include_distances=True, include_species=True),每个位点生成3个特征:

1. site #{i} min. rel. dist. - 位点i到其最近邻的相对距离
2. site #{i} specie - 位点i的元素种类
3. site #{i} neighbor specie(s) - 位点i的最近邻元素种类

如果结构的位点数少于拟合时遇到的最大位点数,输出会用NaN值填充。如果位点数多于最大位点数,结果会被截断。

In [ ]:
# 直接对dataframe进行特征化
df = df_raw.copy()
mrd = MinimumRelativeDistances(
    cutoff=15,            # 搜索近邻的最大距离
    flatten=True,           # 返回固定长度的特征向量
    include_distances=True, # 包含数值距离
    include_species=False    # 包含元素种类信息
)
df = mrd.fit_featurize_dataframe(df, "oxi_structure")
df

MinimumRelativeDistances:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,site #0 min. rel. dist.,site #1 min. rel. dist.,site #2 min. rel. dist.,...,site #28 min. rel. dist.,site #29 min. rel. dist.,site #30 min. rel. dist.,site #31 min. rel. dist.,site #32 min. rel. dist.,site #33 min. rel. dist.,site #34 min. rel. dist.,site #35 min. rel. dist.,site #36 min. rel. dist.,site #37 min. rel. dist.
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,2.242520,2.242435,2.241448,...,2.241683,2.241696,2.241603,2.241892,2.244349,2.242435,2.244336,2.242205,2.244246,2.242456
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.873145,0.873145,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.854622,0.854622,0.817035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Composite

Structure featurizers producing more than one kind of structure feature data.

**JarvisCFID:**

Classical Force-Field Inspired Descriptors (CFID) from Jarvis-ML.

JarvisCFID(Classical Force-Field Inspired Descriptors)是一个强大的结构特征提取器，源自NIST的Jarvis项目。它能够生成多达1,557个特征，**全面描述晶体结构的化学和结构性质**

In [113]:
from matminer.featurizers.structure.composite import JarvisCFID

In [114]:
cfid = JarvisCFID()
# 提取特征
features = cfid.featurize(structure)

# 获取特征标签
feature_labels = cfid.feature_labels()

# 创建包含特征的数据框
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,jml_bp_mult_atom_rad,jml_hfus_add_bp,jml_elec_aff_mult_voro_coord,jml_mol_vol_subs_atom_mass,jml_is_halogen,jml_atom_mass_subs_first_ion_en,jml_row,jml_mol_vol_mult_atom_mass,jml_voro_coord_divi_therm_cond,jml_voro_coord_subs_mp,...,jml_nn_91,jml_nn_92,jml_nn_93,jml_nn_94,jml_nn_95,jml_nn_96,jml_nn_97,jml_nn_98,jml_nn_99,jml_nn_100
0,1159.955,697.584,10.148,-8.636385,0.5,20.168032,3.0,581.612192,224.761958,-263.235,...,0.0,0.0,24.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0


In [118]:
feat_df.isna().sum(axis=1)  # 检查缺失值情况

0    0
dtype: int64

In [119]:
# 直接对dataframe进行特征化
df = df_raw.copy()
df = cfid.fit_featurize_dataframe(df, "oxi_structure")
df

JarvisCFID:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,jml_bp_mult_atom_rad,jml_hfus_add_bp,jml_elec_aff_mult_voro_coord,...,jml_nn_91,jml_nn_92,jml_nn_93,jml_nn_94,jml_nn_95,jml_nn_96,jml_nn_97,jml_nn_98,jml_nn_99,jml_nn_100
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,3250.150000,3737.33805,5.6485,...,1.894737,0.0,26.526316,0.0,22.631579,12.0,1.263158,20.210526,4.421053,24.947368
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",1159.955000,697.58400,10.1480,...,0.000000,0.0,24.000000,0.0,0.000000,0.0,8.000000,0.000000,0.000000,0.000000
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",2075.573333,2049.48720,3.0438,...,7.200000,6.0,13.200000,4.8,4.800000,16.8,21.600000,7.200000,12.000000,13.200000


In [120]:
# 只使用部分特征类型
custom_cfid = JarvisCFID(
    use_cell=True,    # 4个特征：密度、体积等
    use_chem=False,    # 438个特征：基于化学元素性质
    use_chg=False,    # 378个特征：核心电荷分布
    use_rdf=False,     # 100个特征：径向分布函数
    use_adf=False,    # 358个特征(2x179)：角分布函数
    use_ddf=False,    # 179个特征：二面角分布函数
    use_nn=False      # 100个特征：最近邻居分布
)
df = df_raw.copy()
df = custom_cfid.fit_featurize_dataframe(df, "oxi_structure")
df

JarvisCFID:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,jml_pack_frac,jml_vpa,jml_density,jml_log_vpa
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,2.33352,3.227670,10.314143,0.339923
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",3.08249,2.224545,21.812663,0.655991
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",2.50919,2.703520,12.294951,0.464908


In [121]:
# 大规模处理并行化
# 对于大型数据集，可以启用并行计算
# 创建多特征提取器
# JarvisCFID计算量较大，适合单独处理
cfid_subset = JarvisCFID(use_adf=False, use_ddf=False)  # 减少计算负担的配置

# 设置并行处理
cfid_subset.set_n_jobs(-1)  # 使用所有可用CPU核心
df = df_raw.copy()
df = custom_cfid.fit_featurize_dataframe(df, "oxi_structure")
df

JarvisCFID:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,jml_pack_frac,jml_vpa,jml_density,jml_log_vpa
0,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,2.33352,3.227670,10.314143,0.339923
1,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",3.08249,2.224545,21.812663,0.655991
2,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",2.50919,2.703520,12.294951,0.464908


In [ ]:
# 特征筛选和降维
# 由于JarvisCFID生成的特征数量庞大，常配合特征选择或降维方法使用：
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

# 方法1：使用PCA降维
pca = PCA(n_components=50)
features_pca = pca.fit_transform(df_featurized[cfid.feature_labels()])

# 方法2：移除低方差特征
selector = VarianceThreshold(threshold=0.01)
selected_features = selector.fit_transform(df_featurized[cfid.feature_labels()])

**注意事项和常见问题**

1. 计算开销：生成所有1,557个特征对大型结构可能很耗时，考虑禁用一些特征类型或并行处理。

2. 元素兼容性：某些罕见元素可能在内部的元素描述符文件中缺失。如果遇到错误，可以使用ignore_errors=True跳过这些结构。

3. 内存考虑：处理大量结构时，可能需要批量处理以避免内存问题。

## Matrix

In [1]:
from matminer.featurizers.structure.matrix import CoulombMatrix, SineCoulombMatrix, OrbitalFieldMatrix

### 1. CoulombMatrix
用于生成结构的库仑矩阵，该矩阵表示原子核之间的库仑相互作用。

In [9]:
cm = CoulombMatrix(
    diag_elems=True, # 是否对角线元素归一化
    flatten=True, # 是否展平为一维数组
)

In [11]:
# 拟合
cm.fit([structure])
# 特征化
features = cm.featurize(structure)
# 获取特征标签
feature_labels = cm.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,coulomb matrix eig 0,coulomb matrix eig 1,coulomb matrix eig 2,coulomb matrix eig 3,coulomb matrix eig 4,coulomb matrix eig 5,coulomb matrix eig 6,coulomb matrix eig 7,coulomb matrix eig 8,coulomb matrix eig 9,coulomb matrix eig 10,coulomb matrix eig 11,coulomb matrix eig 12,coulomb matrix eig 13,coulomb matrix eig 14,coulomb matrix eig 15,coulomb matrix eig 16,coulomb matrix eig 17,coulomb matrix eig 18,coulomb matrix eig 19,coulomb matrix eig 20,coulomb matrix eig 21,coulomb matrix eig 22,coulomb matrix eig 23,coulomb matrix eig 24,coulomb matrix eig 25,coulomb matrix eig 26,coulomb matrix eig 27,coulomb matrix eig 28,coulomb matrix eig 29,coulomb matrix eig 30,coulomb matrix eig 31,coulomb matrix eig 32,coulomb matrix eig 33,coulomb matrix eig 34,coulomb matrix eig 35,coulomb matrix eig 36,coulomb matrix eig 37
0,530.878122,376.97105,377.024,309.495247,309.566364,282.234609,283.325416,263.930621,263.559939,253.729347,253.415412,246.929972,246.015221,241.815147,242.232227,238.394129,239.335667,239.540434,238.61471,42.788996,38.423401,38.412776,34.979709,34.965574,33.199952,33.076768,31.521428,31.566949,30.573765,30.617069,29.92444,29.792578,29.340369,29.280826,28.870504,29.016914,28.985266,28.909057


In [12]:
feat_df.shape

(1, 38)

In [13]:
df = df_raw.copy()
# 直接对dataframe进行特征化
cm = CoulombMatrix(diag_elems=True, flatten=True)
df = cm.fit_featurize_dataframe(df, "oxi_structure")
df

CoulombMatrix:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,coulomb matrix eig 0,coulomb matrix eig 1,coulomb matrix eig 2,coulomb matrix eig 3,coulomb matrix eig 4,coulomb matrix eig 5,coulomb matrix eig 6,coulomb matrix eig 7,coulomb matrix eig 8,coulomb matrix eig 9,coulomb matrix eig 10,coulomb matrix eig 11,coulomb matrix eig 12,coulomb matrix eig 13,coulomb matrix eig 14,coulomb matrix eig 15,coulomb matrix eig 16,coulomb matrix eig 17,coulomb matrix eig 18,coulomb matrix eig 19,coulomb matrix eig 20,coulomb matrix eig 21,coulomb matrix eig 22,coulomb matrix eig 23,coulomb matrix eig 24,coulomb matrix eig 25,coulomb matrix eig 26,coulomb matrix eig 27,coulomb matrix eig 28,coulomb matrix eig 29,coulomb matrix eig 30,coulomb matrix eig 31,coulomb matrix eig 32,coulomb matrix eig 33,coulomb matrix eig 34,coulomb matrix eig 35,coulomb matrix eig 36,coulomb matrix eig 37
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",153.625252,453.043801,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",744.002535,610.582459,125.450362,21.677926,23.211331,82.193153,59.066093,59.066101,57.674016,57.674010,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,530.878122,376.971050,377.024000,309.495247,309.566364,282.234609,283.325416,263.930621,263.559939,253.729347,253.415412,246.929972,246.015221,241.815147,242.232227,238.394129,239.335667,239.540434,238.61471,42.788996,38.423401,38.412776,34.979709,34.965574,33.199952,33.076768,31.521428,31.566949,30.573765,30.617069,29.92444,29.792578,29.340369,29.280826,28.870504,29.016914,28.985266,28.909057


### 2. SineCoulombMatrix
用于生成周期性晶体的正弦库仑矩阵变体

In [16]:
scm = SineCoulombMatrix(
    diag_elems=True, # 是否对角线元素归一化
    flatten=True # 是否展平为一维数组
)
# 拟合
scm.fit([structure])
# 特征化
features = scm.featurize(structure)
# 获取特征标签
feature_labels = scm.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,sine coulomb matrix eig 0,sine coulomb matrix eig 1,sine coulomb matrix eig 2,sine coulomb matrix eig 3,sine coulomb matrix eig 4,sine coulomb matrix eig 5,sine coulomb matrix eig 6,sine coulomb matrix eig 7,sine coulomb matrix eig 8,sine coulomb matrix eig 9,sine coulomb matrix eig 10,sine coulomb matrix eig 11,sine coulomb matrix eig 12,sine coulomb matrix eig 13,sine coulomb matrix eig 14,sine coulomb matrix eig 15,sine coulomb matrix eig 16,sine coulomb matrix eig 17,sine coulomb matrix eig 18,sine coulomb matrix eig 19,sine coulomb matrix eig 20,sine coulomb matrix eig 21,sine coulomb matrix eig 22,sine coulomb matrix eig 23,sine coulomb matrix eig 24,sine coulomb matrix eig 25,sine coulomb matrix eig 26,sine coulomb matrix eig 27,sine coulomb matrix eig 28,sine coulomb matrix eig 29,sine coulomb matrix eig 30,sine coulomb matrix eig 31,sine coulomb matrix eig 32,sine coulomb matrix eig 33,sine coulomb matrix eig 34,sine coulomb matrix eig 35,sine coulomb matrix eig 36,sine coulomb matrix eig 37
0,462.059032,379.92773,380.166676,323.674496,324.87514,293.325923,286.080537,256.785753,256.15285,260.509763,260.038067,271.182803,266.803695,266.312734,266.116216,269.662835,269.540224,268.960868,269.040257,19.164249,19.248564,19.427666,19.506251,19.86523,19.754527,21.464306,24.864735,22.119908,22.282699,22.356131,24.578501,24.496579,22.844392,22.989978,23.445588,24.06935,23.754736,23.804984


In [17]:
# 直接对dataframe进行特征化
df = df_raw.copy()
scm = SineCoulombMatrix(diag_elems=True, flatten=True)
df = scm.fit_featurize_dataframe(df, "oxi_structure")
df

SineCoulombMatrix:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,sine coulomb matrix eig 0,sine coulomb matrix eig 1,sine coulomb matrix eig 2,sine coulomb matrix eig 3,sine coulomb matrix eig 4,sine coulomb matrix eig 5,sine coulomb matrix eig 6,sine coulomb matrix eig 7,sine coulomb matrix eig 8,sine coulomb matrix eig 9,sine coulomb matrix eig 10,sine coulomb matrix eig 11,sine coulomb matrix eig 12,sine coulomb matrix eig 13,sine coulomb matrix eig 14,sine coulomb matrix eig 15,sine coulomb matrix eig 16,sine coulomb matrix eig 17,sine coulomb matrix eig 18,sine coulomb matrix eig 19,sine coulomb matrix eig 20,sine coulomb matrix eig 21,sine coulomb matrix eig 22,sine coulomb matrix eig 23,sine coulomb matrix eig 24,sine coulomb matrix eig 25,sine coulomb matrix eig 26,sine coulomb matrix eig 27,sine coulomb matrix eig 28,sine coulomb matrix eig 29,sine coulomb matrix eig 30,sine coulomb matrix eig 31,sine coulomb matrix eig 32,sine coulomb matrix eig 33,sine coulomb matrix eig 34,sine coulomb matrix eig 35,sine coulomb matrix eig 36,sine coulomb matrix eig 37
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",157.515820,449.153234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",677.597689,650.432526,35.624444,90.808429,35.416257,72.886764,72.886765,68.258048,68.343533,68.343532,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,462.059032,379.927730,380.166676,323.674496,324.875140,293.325923,286.080537,256.785753,256.152850,260.509763,260.038067,271.182803,266.803695,266.312734,266.116216,269.662835,269.540224,268.960868,269.040257,19.164249,19.248564,19.427666,19.506251,19.86523,19.754527,21.464306,24.864735,22.119908,22.282699,22.356131,24.578501,24.496579,22.844392,22.989978,23.445588,24.06935,23.754736,23.804984


### 3. OrbitalFieldMatrix
用于基于相邻原子的价层电子生成表示。

In [20]:
ofm = OrbitalFieldMatrix(
    period_tag=False, # 是否包含周期标签
    flatten=True, # 是否展平为一维数组
)
# 特征化
features = ofm.featurize(structure) 
# 获取特征标签
feature_labels = ofm.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,OFM: s^1 - s^1,OFM: s^1 - s^2,OFM: s^1 - p^1,OFM: s^1 - p^2,OFM: s^1 - p^3,OFM: s^1 - p^4,OFM: s^1 - p^5,OFM: s^1 - p^6,OFM: s^1 - d^1,OFM: s^1 - d^2,OFM: s^1 - d^3,OFM: s^1 - d^4,OFM: s^1 - d^5,OFM: s^1 - d^6,OFM: s^1 - d^7,OFM: s^1 - d^8,OFM: s^1 - d^9,OFM: s^1 - d^10,OFM: s^1 - f^1,OFM: s^1 - f^2,OFM: s^1 - f^3,OFM: s^1 - f^4,OFM: s^1 - f^5,OFM: s^1 - f^6,OFM: s^1 - f^7,OFM: s^1 - f^8,OFM: s^1 - f^9,OFM: s^1 - f^10,OFM: s^1 - f^11,OFM: s^1 - f^12,OFM: s^1 - f^13,OFM: s^1 - f^14,OFM: s^2 - s^1,OFM: s^2 - s^2,OFM: s^2 - p^1,OFM: s^2 - p^2,OFM: s^2 - p^3,OFM: s^2 - p^4,OFM: s^2 - p^5,OFM: s^2 - p^6,...,OFM: f^13 - f^7,OFM: f^13 - f^8,OFM: f^13 - f^9,OFM: f^13 - f^10,OFM: f^13 - f^11,OFM: f^13 - f^12,OFM: f^13 - f^13,OFM: f^13 - f^14,OFM: f^14 - s^1,OFM: f^14 - s^2,OFM: f^14 - p^1,OFM: f^14 - p^2,OFM: f^14 - p^3,OFM: f^14 - p^4,OFM: f^14 - p^5,OFM: f^14 - p^6,OFM: f^14 - d^1,OFM: f^14 - d^2,OFM: f^14 - d^3,OFM: f^14 - d^4,OFM: f^14 - d^5,OFM: f^14 - d^6,OFM: f^14 - d^7,OFM: f^14 - d^8,OFM: f^14 - d^9,OFM: f^14 - d^10,OFM: f^14 - f^1,OFM: f^14 - f^2,OFM: f^14 - f^3,OFM: f^14 - f^4,OFM: f^14 - f^5,OFM: f^14 - f^6,OFM: f^14 - f^7,OFM: f^14 - f^8,OFM: f^14 - f^9,OFM: f^14 - f^10,OFM: f^14 - f^11,OFM: f^14 - f^12,OFM: f^14 - f^13,OFM: f^14 - f^14
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.195684,0.0,1.195684,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
# 直接对dataframe进行特征化
df = df_raw.copy()
ofm = OrbitalFieldMatrix(period_tag=False, flatten=True)
df = ofm.fit_featurize_dataframe(df, "oxi_structure")
df

OrbitalFieldMatrix:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,OFM: s^1 - s^1,OFM: s^1 - s^2,OFM: s^1 - p^1,OFM: s^1 - p^2,OFM: s^1 - p^3,OFM: s^1 - p^4,OFM: s^1 - p^5,OFM: s^1 - p^6,OFM: s^1 - d^1,OFM: s^1 - d^2,OFM: s^1 - d^3,OFM: s^1 - d^4,OFM: s^1 - d^5,OFM: s^1 - d^6,OFM: s^1 - d^7,OFM: s^1 - d^8,OFM: s^1 - d^9,OFM: s^1 - d^10,OFM: s^1 - f^1,OFM: s^1 - f^2,OFM: s^1 - f^3,OFM: s^1 - f^4,OFM: s^1 - f^5,OFM: s^1 - f^6,OFM: s^1 - f^7,OFM: s^1 - f^8,OFM: s^1 - f^9,OFM: s^1 - f^10,OFM: s^1 - f^11,OFM: s^1 - f^12,OFM: s^1 - f^13,OFM: s^1 - f^14,OFM: s^2 - s^1,...,OFM: f^13 - f^7,OFM: f^13 - f^8,OFM: f^13 - f^9,OFM: f^13 - f^10,OFM: f^13 - f^11,OFM: f^13 - f^12,OFM: f^13 - f^13,OFM: f^13 - f^14,OFM: f^14 - s^1,OFM: f^14 - s^2,OFM: f^14 - p^1,OFM: f^14 - p^2,OFM: f^14 - p^3,OFM: f^14 - p^4,OFM: f^14 - p^5,OFM: f^14 - p^6,OFM: f^14 - d^1,OFM: f^14 - d^2,OFM: f^14 - d^3,OFM: f^14 - d^4,OFM: f^14 - d^5,OFM: f^14 - d^6,OFM: f^14 - d^7,OFM: f^14 - d^8,OFM: f^14 - d^9,OFM: f^14 - d^10,OFM: f^14 - f^1,OFM: f^14 - f^2,OFM: f^14 - f^3,OFM: f^14 - f^4,OFM: f^14 - f^5,OFM: f^14 - f^6,OFM: f^14 - f^7,OFM: f^14 - f^8,OFM: f^14 - f^9,OFM: f^14 - f^10,OFM: f^14 - f^11,OFM: f^14 - f^12,OFM: f^14 - f^13,OFM: f^14 - f^14
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",8.933656e-08,0.568172,0.0,0.0,0.0,0.0,0.568172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.568172,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Misc

In [22]:
from matminer.featurizers.structure.misc import EwaldEnergy, StructureComposition, XRDPowderPattern

### 1. EwaldEnergy
用于计算结构的库仑相互作用能。
Ewald 能量指的是：

在周期性体系（如晶体）中计算离子间库伦相互作用时，用 Ewald Summation（埃瓦尔德求和法） 得到的总电势能。

In [28]:
ee = EwaldEnergy(
    accuracy=4, # 精度
    per_atom=True, # 是否返回每个原子的能量
)
# 特征化
features = ee.featurize(structure)
# 获取特征标签
feature_labels = ee.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,ewald_energy_per_atom
0,-4.503149


In [29]:
# 直接对dataframe进行特征化
df = df_raw.copy()
ee = EwaldEnergy(accuracy=4, per_atom=True)
df = ee.fit_featurize_dataframe(df, "oxi_structure")
df

EwaldEnergy:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,ewald_energy_per_atom
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",-4.503149
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",-45.229812
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,-100.163869


### 2. StructureComposition
用于提取与结构组成相关的特征。它本质上是一个包装器，它在结构的成分上调用基于成分的特征器。

In [30]:
from matminer.featurizers.composition import ElementProperty
ep = ElementProperty.from_preset(preset_name="magpie")
sc = StructureComposition(featurizer=ep)
# 特征化
features = sc.featurize(structure)
# 获取特征标签
feature_labels = sc.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

e:\software2\Anaconda\Lib\site-packages\matminer\utils\data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,MagpieData avg_dev MendeleevNumber,MagpieData mode MendeleevNumber,MagpieData minimum AtomicWeight,MagpieData maximum AtomicWeight,MagpieData range AtomicWeight,MagpieData mean AtomicWeight,MagpieData avg_dev AtomicWeight,MagpieData mode AtomicWeight,MagpieData minimum MeltingT,MagpieData maximum MeltingT,MagpieData range MeltingT,MagpieData mean MeltingT,MagpieData avg_dev MeltingT,MagpieData mode MeltingT,MagpieData minimum Column,MagpieData maximum Column,MagpieData range Column,MagpieData mean Column,MagpieData avg_dev Column,MagpieData mode Column,MagpieData minimum Row,MagpieData maximum Row,MagpieData range Row,MagpieData mean Row,MagpieData avg_dev Row,MagpieData mode Row,MagpieData minimum CovalentRadius,MagpieData maximum CovalentRadius,MagpieData range CovalentRadius,MagpieData mean CovalentRadius,...,MagpieData range NdUnfilled,MagpieData mean NdUnfilled,MagpieData avg_dev NdUnfilled,MagpieData mode NdUnfilled,MagpieData minimum NfUnfilled,MagpieData maximum NfUnfilled,MagpieData range NfUnfilled,MagpieData mean NfUnfilled,MagpieData avg_dev NfUnfilled,MagpieData mode NfUnfilled,MagpieData minimum NUnfilled,MagpieData maximum NUnfilled,MagpieData range NUnfilled,MagpieData mean NUnfilled,MagpieData avg_dev NUnfilled,MagpieData mode NUnfilled,MagpieData minimum GSvolume_pa,MagpieData maximum GSvolume_pa,MagpieData range GSvolume_pa,MagpieData mean GSvolume_pa,MagpieData avg_dev GSvolume_pa,MagpieData mode GSvolume_pa,MagpieData minimum GSbandgap,MagpieData maximum GSbandgap,MagpieData range GSbandgap,MagpieData mean GSbandgap,MagpieData avg_dev GSbandgap,MagpieData mode GSbandgap,MagpieData minimum GSmagmom,MagpieData maximum GSmagmom,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,11.0,17.0,6.0,14.0,3.0,11.0,2.0,94.0,92.0,48.0,46.0,2.0,22.989769,35.453,12.463231,29.221385,6.231615,22.989769,171.6,370.87,199.27,271.235,99.635,171.6,1.0,17.0,16.0,9.0,8.0,1.0,3.0,3.0,0.0,3.0,0.0,3.0,102.0,166.0,64.0,134.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,24.4975,29.243333,4.745833,26.870417,2.372917,24.4975,0.0,2.493,2.493,1.2465,1.2465,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64.0,229.0,165.0,146.5,82.5,64.0


In [33]:
# 直接对dataframe进行特征化
df = df_raw.copy()
ep = ElementProperty.from_preset(preset_name="magpie")
sc = StructureComposition(featurizer=ep)
df = sc.featurize_dataframe(df, "oxi_structure")
df

e:\software2\Anaconda\Lib\site-packages\matminer\utils\data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


StructureComposition:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,MagpieData avg_dev MendeleevNumber,MagpieData mode MendeleevNumber,MagpieData minimum AtomicWeight,MagpieData maximum AtomicWeight,MagpieData range AtomicWeight,MagpieData mean AtomicWeight,MagpieData avg_dev AtomicWeight,MagpieData mode AtomicWeight,MagpieData minimum MeltingT,MagpieData maximum MeltingT,MagpieData range MeltingT,MagpieData mean MeltingT,MagpieData avg_dev MeltingT,MagpieData mode MeltingT,MagpieData minimum Column,MagpieData maximum Column,MagpieData range Column,MagpieData mean Column,MagpieData avg_dev Column,MagpieData mode Column,MagpieData minimum Row,MagpieData maximum Row,MagpieData range Row,...,MagpieData range NdUnfilled,MagpieData mean NdUnfilled,MagpieData avg_dev NdUnfilled,MagpieData mode NdUnfilled,MagpieData minimum NfUnfilled,MagpieData maximum NfUnfilled,MagpieData range NfUnfilled,MagpieData mean NfUnfilled,MagpieData avg_dev NfUnfilled,MagpieData mode NfUnfilled,MagpieData minimum NUnfilled,MagpieData maximum NUnfilled,MagpieData range NUnfilled,MagpieData mean NUnfilled,MagpieData avg_dev NUnfilled,MagpieData mode NUnfilled,MagpieData minimum GSvolume_pa,MagpieData maximum GSvolume_pa,MagpieData range GSvolume_pa,MagpieData mean GSvolume_pa,MagpieData avg_dev GSvolume_pa,MagpieData mode GSvolume_pa,MagpieData minimum GSbandgap,MagpieData maximum GSbandgap,MagpieData range GSbandgap,MagpieData mean GSbandgap,MagpieData avg_dev GSbandgap,MagpieData mode GSbandgap,MagpieData minimum GSmagmom,MagpieData maximum GSmagmom,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",11.0,17.0,6.0,14.0,3.0,11.0,2.0,94.0,92.0,48.0,46.0,2.0,22.989769,35.4530,12.463231,29.221385,6.231615,22.989769,171.6,370.87,199.27,271.235,99.635,171.6,1.0,17.0,16.0,9.0,8.00,1.0,3.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,24.4975,29.243333,4.745833,26.870417,2.372917,24.4975,0.000,2.493,2.493,1.2465,1.24650,0.000,0.0,0.0,0.0,0.0,0.0,0.0,64.0,229.0,165.0,146.5,82.5,64.0
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",6.0,20.0,14.0,10.0,4.0,8.0,7.0,87.0,80.0,69.0,24.8,87.0,12.010700,40.0780,28.067300,20.017380,8.024248,15.999400,54.8,3823.00,3768.20,1020.480,1158.816,54.8,2.0,16.0,14.0,12.8,4.32,16.0,2.0,4.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,2.0,0.8,2.0,5.6400,37.770000,32.130000,14.145000,9.450000,9.1050,0.000,4.496,4.496,0.8992,1.43872,0.000,0.0,0.0,0.0,0.0,0.0,0.0,12.0,225.0,213.0,91.0,94.8,12.0
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,6.0,14.0,8.0,10.0,4.0,6.0,77.0,78.0,1.0,77.5,0.5,77.0,12.010700,28.0855,16.074800,20.048100,8.037400,12.010700,1687.0,3823.00,2136.00,2755.000,1068.000,1687.0,14.0,14.0,0.0,14.0,0.00,14.0,2.0,3.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,0.0,4.0,0.0,4.0,5.6400,20.440000,14.800000,13.040000,7.400000,5.6400,0.773,4.496,3.723,2.6345,1.86150,0.773,0.0,0.0,0.0,0.0,0.0,0.0,194.0,227.0,33.0,210.5,16.5,194.0


### 3. XRDPowderPattern
用于生成结构的粉末衍射图。

In [34]:
xrd = XRDPowderPattern(
    two_theta_range=(0, 90), # 2θ范围
    bw_method=0.05
)
# 特征化
features = xrd.featurize(structure)
# 获取特征标签
feature_labels = xrd.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,xrd_0,xrd_1,xrd_2,xrd_3,xrd_4,xrd_5,xrd_6,xrd_7,xrd_8,xrd_9,xrd_10,xrd_11,xrd_12,xrd_13,xrd_14,xrd_15,xrd_16,xrd_17,xrd_18,xrd_19,xrd_20,xrd_21,xrd_22,xrd_23,xrd_24,xrd_25,xrd_26,xrd_27,xrd_28,xrd_29,xrd_30,xrd_31,xrd_32,xrd_33,xrd_34,xrd_35,xrd_36,xrd_37,xrd_38,xrd_39,...,xrd_51,xrd_52,xrd_53,xrd_54,xrd_55,xrd_56,xrd_57,xrd_58,xrd_59,xrd_60,xrd_61,xrd_62,xrd_63,xrd_64,xrd_65,xrd_66,xrd_67,xrd_68,xrd_69,xrd_70,xrd_71,xrd_72,xrd_73,xrd_74,xrd_75,xrd_76,xrd_77,xrd_78,xrd_79,xrd_80,xrd_81,xrd_82,xrd_83,xrd_84,xrd_85,xrd_86,xrd_87,xrd_88,xrd_89,xrd_90
0,1.498543e-212,1.233416e-197,2.865212e-183,1.878499e-169,3.475936e-156,1.815263e-143,2.675556e-131,1.113001e-119,1.306723e-108,4.329908e-98,4.049305e-88,1.068782e-78,7.961677e-70,1.673890e-61,9.932439e-54,1.663381e-46,7.862027e-40,1.048779e-33,3.948575e-28,4.195692e-23,1.258269e-18,1.065002e-14,2.544099e-11,1.715238e-08,0.000003,0.000175,0.002657,0.011365,0.013727,0.005227,0.014029,0.094431,0.185288,0.102622,0.016041,0.000708,0.000009,3.096623e-08,3.071265e-11,1.584248e-14,...,9.713641e-07,0.000041,0.000484,0.00171,0.003842,0.017754,0.036898,0.022121,0.003746,0.000179,0.000002,1.168412e-08,7.250200e-07,0.00006,0.001381,0.009035,0.016685,0.008696,0.001279,0.000053,6.241036e-07,6.745118e-07,0.000072,0.002147,0.018193,0.043504,0.02936,0.005592,0.000301,0.000005,0.000001,0.000091,0.002267,0.015903,0.031489,0.017597,0.002775,0.000124,0.000002,5.502921e-09


In [36]:
# 直接对dataframe进行特征化
df = df_raw.copy()
xrd = XRDPowderPattern(two_theta_range=(0, 90), bw_method=0.05)
df = xrd.featurize_dataframe(df, "oxi_structure")
df

XRDPowderPattern:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,xrd_0,xrd_1,xrd_2,xrd_3,xrd_4,xrd_5,xrd_6,xrd_7,xrd_8,xrd_9,xrd_10,xrd_11,xrd_12,xrd_13,xrd_14,xrd_15,xrd_16,xrd_17,xrd_18,xrd_19,xrd_20,xrd_21,xrd_22,xrd_23,xrd_24,xrd_25,xrd_26,xrd_27,xrd_28,xrd_29,xrd_30,xrd_31,xrd_32,...,xrd_51,xrd_52,xrd_53,xrd_54,xrd_55,xrd_56,xrd_57,xrd_58,xrd_59,xrd_60,xrd_61,xrd_62,xrd_63,xrd_64,xrd_65,xrd_66,xrd_67,xrd_68,xrd_69,xrd_70,xrd_71,xrd_72,xrd_73,xrd_74,xrd_75,xrd_76,xrd_77,xrd_78,xrd_79,xrd_80,xrd_81,xrd_82,xrd_83,xrd_84,xrd_85,xrd_86,xrd_87,xrd_88,xrd_89,xrd_90
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",1.498543e-212,1.233416e-197,2.865212e-183,1.878499e-169,3.475936e-156,1.815263e-143,2.675556e-131,1.113001e-119,1.306723e-108,4.329908e-98,4.049305e-88,1.068782e-78,7.961677e-70,1.673890e-61,9.932439e-54,1.663381e-46,7.862027e-40,1.048779e-33,3.948575e-28,4.195692e-23,1.258269e-18,1.065002e-14,2.544099e-11,1.715238e-08,3.263787e-06,1.752774e-04,2.656665e-03,1.136463e-02,1.372710e-02,5.226501e-03,1.402892e-02,0.094431,0.185288,...,9.713641e-07,4.055400e-05,4.838761e-04,0.001710,0.003842,0.017754,0.036898,0.022121,0.003746,0.000179,0.000002,1.168412e-08,7.250200e-07,0.000060,0.001381,0.009035,0.016685,0.008696,0.001279,0.000053,6.241036e-07,6.745118e-07,0.000072,0.002147,0.018193,0.043504,0.029360,0.005592,3.006380e-04,4.564707e-06,1.054972e-06,9.119521e-05,2.266865e-03,0.015903,0.031489,0.017597,0.002775,1.235370e-04,1.552000e-06,5.502921e-09
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",7.173255e-212,4.401992e-194,4.385537e-177,7.093110e-161,1.862478e-145,7.939367e-131,5.494412e-117,6.173010e-104,1.125935e-91,3.334033e-80,1.602754e-69,1.250848e-59,1.584830e-50,3.259876e-42,1.088579e-34,5.901461e-28,5.193970e-22,7.421297e-17,1.721475e-12,6.482789e-09,3.963361e-06,3.933741e-04,6.338528e-03,1.658103e-02,7.041661e-03,4.854922e-04,9.899875e-06,9.310976e-04,3.151522e-02,1.731858e-01,1.548993e-01,0.025378,0.004103,...,1.825812e-04,8.332426e-07,2.228994e-07,0.000045,0.001590,0.012175,0.025883,0.014537,0.003537,0.010419,0.016694,8.209979e-03,5.867210e-03,0.014930,0.016944,0.011646,0.003811,0.000266,0.000918,0.004761,4.117552e-03,1.858348e-03,0.002757,0.000967,0.000593,0.002004,0.004059,0.003175,4.630441e-04,1.370426e-04,1.004036e-03,2.600586e-03,9.985438e-03,0.013210,0.005599,0.000861,0.000029,1.617967e-07,1.496136e-10,2.248082e-14
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.142408e-310,4.465792e-289,2.860536e-268,3.003536e-248,5.171615e-229,1.460843e-210,6.772390e-193,5.154833e-176,6.444475e-160,1.323779e-144,4.469294e-130,2.480762e-116,2.264490e-103,3.400242e-91,8.401033e-80,3.416726e-69,2.288735e-59,2.527457e-50,4.607819e-42,1.389910e-34,6.959621e-28,5.812104e-22,8.146636e-17,1.931655e-12,7.815687e-09,0.000005,0.000657,...,2.688170e-07,2.401975e-05,4.459326e-04,0.002971,0.007474,0.004789,0.001403,0.003121,0.029510,0.089206,0.051517,5.661646e-03,3.482689e-04,0.002507,0.013731,0.020497,0.006916,0.000470,0.000141,0.004344,3.680171e-02,7.081319e-02,0.030133,0.007911,0.005664,0.005658,0.001139,0.000038,2.149383e-07,2.001264e-10,2.246765e-13,5.956472e-10,2.990716e-07,0.000025,0.000329,0.000719,0.000257,1.499211e-05,1.429261e-07,2.225551e-10


## Order
Structure featurizers based on packing or ordering.

In [37]:
from matminer.featurizers.structure.order import DensityFeatures, ChemicalOrdering, MaximumPackingEfficiency, StructuralComplexity

### 1. DensityFeatures
用于计算密度和类密度特征

In [39]:
densityFeat = DensityFeatures(desired_features=['density', 'vpa'])
# 特征化
features = densityFeat.featurize(structure)
# 获取特征标签
feature_labels = densityFeat.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,density,vpa
0,2.224545,21.812663


In [40]:
# 直接对dataframe进行特征化
df = df_raw.copy()
densityFeat = DensityFeatures(desired_features=['density', 'vpa'])
df = densityFeat.featurize_dataframe(df, "oxi_structure")
df

DensityFeatures:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,density,vpa
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",2.224545,21.812663
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",2.703520,12.294951
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,3.227670,10.314143


### 2. ChemicalOrdering
用于量化结构中物种排序与随机排序的偏差程度。

In [41]:
co = ChemicalOrdering()

# 特征化
features = co.featurize(structure)
# 获取特征标签
feature_labels = co.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,mean ordering parameter shell 1,mean ordering parameter shell 2,mean ordering parameter shell 3
0,0.999999,0.999999,0.999998


In [43]:
# 直接对dataframe进行特征化
df = df_raw.copy()
co = ChemicalOrdering()
df = co.featurize_dataframe(df, "oxi_structure", ignore_errors=True)
df

ChemicalOrdering:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,mean ordering parameter shell 1,mean ordering parameter shell 2,mean ordering parameter shell 3
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.999999,0.999999,0.999998
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.607061,0.337031,0.120077
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,NaN,NaN,NaN


### 3. MaximumPackingEfficiency
用于计算结构的最大可能堆积效率。

In [44]:
mpe = MaximumPackingEfficiency()
# 特征化
features = mpe.featurize(structure)
# 获取特征标签
feature_labels = mpe.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,max packing efficiency
0,0.523599


In [46]:
# 直接对dataframe进行特征化
df = df_raw.copy()
mpe = MaximumPackingEfficiency()
df = mpe.featurize_dataframe(df, "oxi_structure", ignore_errors=True)
df

MaximumPackingEfficiency:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,max packing efficiency
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.523599
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.185069
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.339431


### 3. StructuralComplexity
用于计算结构的香农信息熵，以此来衡量结构的复杂性

In [48]:
sc = StructuralComplexity()
# 特征化
features = sc.featurize(structure)
# 获取特征标签
feature_labels = sc.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,structural complexity per atom,structural complexity per cell
0,1.0,2.0


In [49]:
# 直接对dataframe进行特征化
df = df_raw.copy()
sc = StructuralComplexity()
df = sc.featurize_dataframe(df, "oxi_structure", ignore_errors=True)
df

StructuralComplexity:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,structural complexity per atom,structural complexity per cell
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",1.000000,2.000000
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",1.370951,13.709506
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,5.247928,199.421246


## RDF

In [51]:
from matminer.featurizers.structure.rdf import RadialDistributionFunction, PartialRadialDistributionFunction, ElectronicRadialDistributionFunction

### 1. RadialDistributionFunction
用于计算晶体结构的径向分布函数 (RDF)。

In [52]:
rdf = RadialDistributionFunction(
    cutoff=10,  # 计算rdf的最大距离（以Å为单位）
    bin_size=0.1,  # 每个bin的大小（以Å为单位）
)
# 特征化
features = rdf.featurize(structure)
# 获取特征标签
feature_labels = rdf.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,rdf [0.00000 - 0.10000]A,rdf [0.10000 - 0.20000]A,rdf [0.20000 - 0.30000]A,rdf [0.30000 - 0.40000]A,rdf [0.40000 - 0.50000]A,rdf [0.50000 - 0.60000]A,rdf [0.60000 - 0.70000]A,rdf [0.70000 - 0.80000]A,rdf [0.80000 - 0.90000]A,rdf [0.90000 - 1.00000]A,rdf [1.00000 - 1.10000]A,rdf [1.10000 - 1.20000]A,rdf [1.20000 - 1.30000]A,rdf [1.30000 - 1.40000]A,rdf [1.40000 - 1.50000]A,rdf [1.50000 - 1.60000]A,rdf [1.60000 - 1.70000]A,rdf [1.70000 - 1.80000]A,rdf [1.80000 - 1.90000]A,rdf [1.90000 - 2.00000]A,rdf [2.00000 - 2.10000]A,rdf [2.10000 - 2.20000]A,rdf [2.20000 - 2.30000]A,rdf [2.30000 - 2.40000]A,rdf [2.40000 - 2.50000]A,rdf [2.50000 - 2.60000]A,rdf [2.60000 - 2.70000]A,rdf [2.70000 - 2.80000]A,rdf [2.80000 - 2.90000]A,rdf [2.90000 - 3.00000]A,rdf [3.00000 - 3.10000]A,rdf [3.10000 - 3.20000]A,rdf [3.20000 - 3.30000]A,rdf [3.30000 - 3.40000]A,rdf [3.40000 - 3.50000]A,rdf [3.50000 - 3.60000]A,rdf [3.60000 - 3.70000]A,rdf [3.70000 - 3.80000]A,rdf [3.80000 - 3.90000]A,rdf [3.90000 - 4.00000]A,...,rdf [6.00000 - 6.10000]A,rdf [6.10000 - 6.20000]A,rdf [6.20000 - 6.30000]A,rdf [6.30000 - 6.40000]A,rdf [6.40000 - 6.50000]A,rdf [6.50000 - 6.60000]A,rdf [6.60000 - 6.70000]A,rdf [6.70000 - 6.80000]A,rdf [6.80000 - 6.90000]A,rdf [6.90000 - 7.00000]A,rdf [7.00000 - 7.10000]A,rdf [7.10000 - 7.20000]A,rdf [7.20000 - 7.30000]A,rdf [7.30000 - 7.40000]A,rdf [7.40000 - 7.50000]A,rdf [7.50000 - 7.60000]A,rdf [7.60000 - 7.70000]A,rdf [7.70000 - 7.80000]A,rdf [7.80000 - 7.90000]A,rdf [7.90000 - 8.00000]A,rdf [8.00000 - 8.10000]A,rdf [8.10000 - 8.20000]A,rdf [8.20000 - 8.30000]A,rdf [8.30000 - 8.40000]A,rdf [8.40000 - 8.50000]A,rdf [8.50000 - 8.60000]A,rdf [8.60000 - 8.70000]A,rdf [8.70000 - 8.80000]A,rdf [8.80000 - 8.90000]A,rdf [8.90000 - 9.00000]A,rdf [9.00000 - 9.10000]A,rdf [9.10000 - 9.20000]A,rdf [9.20000 - 9.30000]A,rdf [9.30000 - 9.40000]A,rdf [9.40000 - 9.50000]A,rdf [9.50000 - 9.60000]A,rdf [9.60000 - 9.70000]A,rdf [9.70000 - 9.80000]A,rdf [9.80000 - 9.90000]A,rdf [9.90000 - 10.00000]A
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.540183,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.698857,...,0.0,0.0,21.329013,0.0,0.0,0.0,0.0,0.0,17.756248,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.591285,0.0,0.0,0.0,14.937294,0.0,0.0,0.0,0.0,10.637728,0.0,0.0,0.0,9.73761,0.0,0.0,0.0,2.982362,0.0,0.0,0.0


In [53]:
# 直接对dataframe进行特征化
df = df_raw.copy()
rdf = RadialDistributionFunction(cutoff=10, bin_size=0.1)
df = rdf.featurize_dataframe(df, "oxi_structure")
df

RadialDistributionFunction:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,rdf [0.00000 - 0.10000]A,rdf [0.10000 - 0.20000]A,rdf [0.20000 - 0.30000]A,rdf [0.30000 - 0.40000]A,rdf [0.40000 - 0.50000]A,rdf [0.50000 - 0.60000]A,rdf [0.60000 - 0.70000]A,rdf [0.70000 - 0.80000]A,rdf [0.80000 - 0.90000]A,rdf [0.90000 - 1.00000]A,rdf [1.00000 - 1.10000]A,rdf [1.10000 - 1.20000]A,rdf [1.20000 - 1.30000]A,rdf [1.30000 - 1.40000]A,rdf [1.40000 - 1.50000]A,rdf [1.50000 - 1.60000]A,rdf [1.60000 - 1.70000]A,rdf [1.70000 - 1.80000]A,rdf [1.80000 - 1.90000]A,rdf [1.90000 - 2.00000]A,rdf [2.00000 - 2.10000]A,rdf [2.10000 - 2.20000]A,rdf [2.20000 - 2.30000]A,rdf [2.30000 - 2.40000]A,rdf [2.40000 - 2.50000]A,rdf [2.50000 - 2.60000]A,rdf [2.60000 - 2.70000]A,rdf [2.70000 - 2.80000]A,rdf [2.80000 - 2.90000]A,rdf [2.90000 - 3.00000]A,rdf [3.00000 - 3.10000]A,rdf [3.10000 - 3.20000]A,rdf [3.20000 - 3.30000]A,...,rdf [6.00000 - 6.10000]A,rdf [6.10000 - 6.20000]A,rdf [6.20000 - 6.30000]A,rdf [6.30000 - 6.40000]A,rdf [6.40000 - 6.50000]A,rdf [6.50000 - 6.60000]A,rdf [6.60000 - 6.70000]A,rdf [6.70000 - 6.80000]A,rdf [6.80000 - 6.90000]A,rdf [6.90000 - 7.00000]A,rdf [7.00000 - 7.10000]A,rdf [7.10000 - 7.20000]A,rdf [7.20000 - 7.30000]A,rdf [7.30000 - 7.40000]A,rdf [7.40000 - 7.50000]A,rdf [7.50000 - 7.60000]A,rdf [7.60000 - 7.70000]A,rdf [7.70000 - 7.80000]A,rdf [7.80000 - 7.90000]A,rdf [7.90000 - 8.00000]A,rdf [8.00000 - 8.10000]A,rdf [8.10000 - 8.20000]A,rdf [8.20000 - 8.30000]A,rdf [8.30000 - 8.40000]A,rdf [8.40000 - 8.50000]A,rdf [8.50000 - 8.60000]A,rdf [8.60000 - 8.70000]A,rdf [8.70000 - 8.80000]A,rdf [8.80000 - 8.90000]A,rdf [8.90000 - 9.00000]A,rdf [9.00000 - 9.10000]A,rdf [9.10000 - 9.20000]A,rdf [9.20000 - 9.30000]A,rdf [9.30000 - 9.40000]A,rdf [9.40000 - 9.50000]A,rdf [9.50000 - 9.60000]A,rdf [9.60000 - 9.70000]A,rdf [9.70000 - 9.80000]A,rdf [9.80000 - 9.90000]A,rdf [9.90000 - 10.00000]A
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,27.540183,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,21.329013,0.000000,0.000000,0.000000,0.000000,0.0,17.756248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,6.591285,0.000000,0.000000,0.000000,14.937294,0.000000,0.000000,0.000000,0.000000,10.637728,0.000000,0.000000,0.000000,9.737610,0.000000,0.000000,0.000000,2.982362,0.000000,0.000000,0.000000
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,75.10115,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,23.187913,42.513505,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,11.831521,44.458743,...,6.415161,12.416476,9.016752,14.558336,14.110424,27.365736,2.654891,0.0,2.502127,17.014497,14.173069,9.186266,4.467302,8.693142,21.153348,12.358028,6.018529,17.59265,7.621018,3.715251,9.058803,17.675736,3.449967,3.367829,2.740491,12.848457,25.106207,4.600432,2.998037,5.862834,8.600978,7.011673,15.093959,5.371936,5.258847,18.022519,22.694058,7.410310,12.101020,13.044909
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,364.433007,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,402.299568,9.925376,0.000000,...,0.000000,98.952859,0.000000,0.000000,151.515267,0.000000,13.363011,0.0,134.337007,14.273336,0.000000,55.870607,0.000000,43.755678,0.000000,47.227711,0.000000,114.78712,0.000000,18.700181,0.000000,173.488122,0.000000,139.849721,0.000000,16.167714,0.000000,47.597642,30.180375,43.

### 2. PartialRadialDistributionFunction
类用于计算晶体结构的局部径向分布函数 (PRDF)。

In [54]:
prdf = PartialRadialDistributionFunction(
    cutoff=10,  # 计算rdf的最大距离（以Å为单位）
    bin_size=0.1,  # 每个bin的大小（以Å为单位）
)
# 拟合
prdf.fit([structure])
# 特征化
features = prdf.featurize(structure)
# 获取特征标签
feature_labels = prdf.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)  
feat_df

,Na-Na PRDF r=0.00-0.10,Na-Na PRDF r=0.10-0.20,Na-Na PRDF r=0.20-0.30,Na-Na PRDF r=0.30-0.40,Na-Na PRDF r=0.40-0.50,Na-Na PRDF r=0.50-0.60,Na-Na PRDF r=0.60-0.70,Na-Na PRDF r=0.70-0.80,Na-Na PRDF r=0.80-0.90,Na-Na PRDF r=0.90-1.00,Na-Na PRDF r=1.00-1.10,Na-Na PRDF r=1.10-1.20,Na-Na PRDF r=1.20-1.30,Na-Na PRDF r=1.30-1.40,Na-Na PRDF r=1.40-1.50,Na-Na PRDF r=1.50-1.60,Na-Na PRDF r=1.60-1.70,Na-Na PRDF r=1.70-1.80,Na-Na PRDF r=1.80-1.90,Na-Na PRDF r=1.90-2.00,Na-Na PRDF r=2.00-2.10,Na-Na PRDF r=2.10-2.20,Na-Na PRDF r=2.20-2.30,Na-Na PRDF r=2.30-2.40,Na-Na PRDF r=2.40-2.50,Na-Na PRDF r=2.50-2.60,Na-Na PRDF r=2.60-2.70,Na-Na PRDF r=2.70-2.80,Na-Na PRDF r=2.80-2.90,Na-Na PRDF r=2.90-3.00,Na-Na PRDF r=3.00-3.10,Na-Na PRDF r=3.10-3.20,Na-Na PRDF r=3.20-3.30,Na-Na PRDF r=3.30-3.40,Na-Na PRDF r=3.40-3.50,Na-Na PRDF r=3.50-3.60,Na-Na PRDF r=3.60-3.70,Na-Na PRDF r=3.70-3.80,Na-Na PRDF r=3.80-3.90,Na-Na PRDF r=3.90-4.00,...,Cl-Cl PRDF r=6.00-6.10,Cl-Cl PRDF r=6.10-6.20,Cl-Cl PRDF r=6.20-6.30,Cl-Cl PRDF r=6.30-6.40,Cl-Cl PRDF r=6.40-6.50,Cl-Cl PRDF r=6.50-6.60,Cl-Cl PRDF r=6.60-6.70,Cl-Cl PRDF r=6.70-6.80,Cl-Cl PRDF r=6.80-6.90,Cl-Cl PRDF r=6.90-7.00,Cl-Cl PRDF r=7.00-7.10,Cl-Cl PRDF r=7.10-7.20,Cl-Cl PRDF r=7.20-7.30,Cl-Cl PRDF r=7.30-7.40,Cl-Cl PRDF r=7.40-7.50,Cl-Cl PRDF r=7.50-7.60,Cl-Cl PRDF r=7.60-7.70,Cl-Cl PRDF r=7.70-7.80,Cl-Cl PRDF r=7.80-7.90,Cl-Cl PRDF r=7.90-8.00,Cl-Cl PRDF r=8.00-8.10,Cl-Cl PRDF r=8.10-8.20,Cl-Cl PRDF r=8.20-8.30,Cl-Cl PRDF r=8.30-8.40,Cl-Cl PRDF r=8.40-8.50,Cl-Cl PRDF r=8.50-8.60,Cl-Cl PRDF r=8.60-8.70,Cl-Cl PRDF r=8.70-8.80,Cl-Cl PRDF r=8.80-8.90,Cl-Cl PRDF r=8.90-9.00,Cl-Cl PRDF r=9.00-9.10,Cl-Cl PRDF r=9.10-9.20,Cl-Cl PRDF r=9.20-9.30,Cl-Cl PRDF r=9.30-9.40,Cl-Cl PRDF r=9.40-9.50,Cl-Cl PRDF r=9.50-9.60,Cl-Cl PRDF r=9.60-9.70,Cl-Cl PRDF r=9.70-9.80,Cl-Cl PRDF r=9.80-9.90,Cl-Cl PRDF r=9.90-10.00
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.612004,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.407017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.151088,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.243843,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.068363,0.0,0.0,0.0


In [55]:
# 直接对dataframe进行特征化
df = df_raw.copy()
prdf = PartialRadialDistributionFunction(cutoff=10, bin_size=0.1)
df = prdf.fit_featurize_dataframe(df, "oxi_structure")
df

PartialRadialDistributionFunction:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,Na-Na PRDF r=0.00-0.10,Na-Na PRDF r=0.10-0.20,Na-Na PRDF r=0.20-0.30,Na-Na PRDF r=0.30-0.40,Na-Na PRDF r=0.40-0.50,Na-Na PRDF r=0.50-0.60,Na-Na PRDF r=0.60-0.70,Na-Na PRDF r=0.70-0.80,Na-Na PRDF r=0.80-0.90,Na-Na PRDF r=0.90-1.00,Na-Na PRDF r=1.00-1.10,Na-Na PRDF r=1.10-1.20,Na-Na PRDF r=1.20-1.30,Na-Na PRDF r=1.30-1.40,Na-Na PRDF r=1.40-1.50,Na-Na PRDF r=1.50-1.60,Na-Na PRDF r=1.60-1.70,Na-Na PRDF r=1.70-1.80,Na-Na PRDF r=1.80-1.90,Na-Na PRDF r=1.90-2.00,Na-Na PRDF r=2.00-2.10,Na-Na PRDF r=2.10-2.20,Na-Na PRDF r=2.20-2.30,Na-Na PRDF r=2.30-2.40,Na-Na PRDF r=2.40-2.50,Na-Na PRDF r=2.50-2.60,Na-Na PRDF r=2.60-2.70,Na-Na PRDF r=2.70-2.80,Na-Na PRDF r=2.80-2.90,Na-Na PRDF r=2.90-3.00,Na-Na PRDF r=3.00-3.10,Na-Na PRDF r=3.10-3.20,Na-Na PRDF r=3.20-3.30,...,O-O PRDF r=6.00-6.10,O-O PRDF r=6.10-6.20,O-O PRDF r=6.20-6.30,O-O PRDF r=6.30-6.40,O-O PRDF r=6.40-6.50,O-O PRDF r=6.50-6.60,O-O PRDF r=6.60-6.70,O-O PRDF r=6.70-6.80,O-O PRDF r=6.80-6.90,O-O PRDF r=6.90-7.00,O-O PRDF r=7.00-7.10,O-O PRDF r=7.10-7.20,O-O PRDF r=7.20-7.30,O-O PRDF r=7.30-7.40,O-O PRDF r=7.40-7.50,O-O PRDF r=7.50-7.60,O-O PRDF r=7.60-7.70,O-O PRDF r=7.70-7.80,O-O PRDF r=7.80-7.90,O-O PRDF r=7.90-8.00,O-O PRDF r=8.00-8.10,O-O PRDF r=8.10-8.20,O-O PRDF r=8.20-8.30,O-O PRDF r=8.30-8.40,O-O PRDF r=8.40-8.50,O-O PRDF r=8.50-8.60,O-O PRDF r=8.60-8.70,O-O PRDF r=8.70-8.80,O-O PRDF r=8.80-8.90,O-O PRDF r=8.90-9.00,O-O PRDF r=9.00-9.10,O-O PRDF r=9.10-9.20,O-O PRDF r=9.20-9.30,O-O PRDF r=9.30-9.40,O-O PRDF r=9.40-9.50,O-O PRDF r=9.50-9.60,O-O PRDF r=9.60-9.70,O-O PRDF r=9.70-9.80,O-O PRDF r=9.80-9.90,O-O PRDF r=9.90-10.00
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.0,0.00000,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.081486,0.118409,0.038255,0.074192,0.035989,0.0,0.033918,0.098847,0.064042,0.0,0.060557,0.0,0.114699,0.055841,0.027195,0.079494,0.051654,0.050363,0.073679,0.047921,0.0,0.045653,0.02229,0.043542,0.127625,0.0,0.04064,0.0,0.038864,0.057029,0.055803,0.03641,0.035644,0.139604,0.102545,0.0,0.032808,0.080379
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.0,0.00000,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000


### 3. ElectronicRadialDistributionFunction
用于计算固有的电子径向分布函数 (ReDF)。

In [62]:
redf = ElectronicRadialDistributionFunction(
    cutoff=10,  # 计算rdf的最大距离（以Å为单位）
    dr=0.05,  # ReDF的bin宽度（以Å为单位）
)
# 特征化
features = redf.featurize(structure)
# 获取特征标签
feature_labels = redf.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,ReDF [0.00000 - 0.05000]A,ReDF [0.05000 - 0.10000]A,ReDF [0.10000 - 0.15000]A,ReDF [0.15000 - 0.20000]A,ReDF [0.20000 - 0.25000]A,ReDF [0.25000 - 0.30000]A,ReDF [0.30000 - 0.35000]A,ReDF [0.35000 - 0.40000]A,ReDF [0.40000 - 0.45000]A,ReDF [0.45000 - 0.50000]A,ReDF [0.50000 - 0.55000]A,ReDF [0.55000 - 0.60000]A,ReDF [0.60000 - 0.65000]A,ReDF [0.65000 - 0.70000]A,ReDF [0.70000 - 0.75000]A,ReDF [0.75000 - 0.80000]A,ReDF [0.80000 - 0.85000]A,ReDF [0.85000 - 0.90000]A,ReDF [0.90000 - 0.95000]A,ReDF [0.95000 - 1.00000]A,ReDF [1.00000 - 1.05000]A,ReDF [1.05000 - 1.10000]A,ReDF [1.10000 - 1.15000]A,ReDF [1.15000 - 1.20000]A,ReDF [1.20000 - 1.25000]A,ReDF [1.25000 - 1.30000]A,ReDF [1.30000 - 1.35000]A,ReDF [1.35000 - 1.40000]A,ReDF [1.40000 - 1.45000]A,ReDF [1.45000 - 1.50000]A,ReDF [1.50000 - 1.55000]A,ReDF [1.55000 - 1.60000]A,ReDF [1.60000 - 1.65000]A,ReDF [1.65000 - 1.70000]A,ReDF [1.70000 - 1.75000]A,ReDF [1.75000 - 1.80000]A,ReDF [1.80000 - 1.85000]A,ReDF [1.85000 - 1.90000]A,ReDF [1.90000 - 1.95000]A,ReDF [1.95000 - 2.00000]A,...,ReDF [8.05000 - 8.10000]A,ReDF [8.10000 - 8.15000]A,ReDF [8.15000 - 8.20000]A,ReDF [8.20000 - 8.25000]A,ReDF [8.25000 - 8.30000]A,ReDF [8.30000 - 8.35000]A,ReDF [8.35000 - 8.40000]A,ReDF [8.40000 - 8.45000]A,ReDF [8.45000 - 8.50000]A,ReDF [8.50000 - 8.55000]A,ReDF [8.55000 - 8.60000]A,ReDF [8.60000 - 8.65000]A,ReDF [8.65000 - 8.70000]A,ReDF [8.70000 - 8.75000]A,ReDF [8.75000 - 8.80000]A,ReDF [8.80000 - 8.85000]A,ReDF [8.85000 - 8.90000]A,ReDF [8.90000 - 8.95000]A,ReDF [8.95000 - 9.00000]A,ReDF [9.00000 - 9.05000]A,ReDF [9.05000 - 9.10000]A,ReDF [9.10000 - 9.15000]A,ReDF [9.15000 - 9.20000]A,ReDF [9.20000 - 9.25000]A,ReDF [9.25000 - 9.30000]A,ReDF [9.30000 - 9.35000]A,ReDF [9.35000 - 9.40000]A,ReDF [9.40000 - 9.45000]A,ReDF [9.45000 - 9.50000]A,ReDF [9.50000 - 9.55000]A,ReDF [9.55000 - 9.60000]A,ReDF [9.60000 - 9.65000]A,ReDF [9.65000 - 9.70000]A,ReDF [9.70000 - 9.75000]A,ReDF [9.75000 - 9.80000]A,ReDF [9.80000 - 9.85000]A,ReDF [9.85000 - 9.90000]A,ReDF [9.90000 - 9.95000]A,ReDF [9.95000 - 10.00000]A,ReDF [10.00000 - 10.00000]A
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-3.579017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.716283,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.589874,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.826539,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
# 直接对dataframe进行特征化
df = df_raw.copy()
redf = ElectronicRadialDistributionFunction(cutoff=10, dr=0.05)
redf.featurize_dataframe(df, "oxi_structure")
df

ElectronicRadialDistributionFunction:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ..."
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ..."
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...


## Sites
Structure featurizers based on aggregating site features.

In [67]:
from matminer.featurizers.structure.sites import SiteStatsFingerprint

### 1. SiteStatsFingerprint
类用于计算结构中所有位点属性的统计信息。
它首先使用位点特征器类（请参阅 site.py 了解选项）来计算结构中每个位点的特征，然后通过测量每个属性的统计信息来计算整个结构的特征。
可以选择仅计算具有一定氧化态范围的位点的统计信息（例如，仅阴离子）。

In [70]:
from matminer.featurizers.site import CoordinationNumber
from pymatgen.analysis.local_env import VoronoiNN, JmolNN, MinimumDistanceNN
ssf = SiteStatsFingerprint(
    site_featurizer=CoordinationNumber(nn=VoronoiNN()), # 使用Voronoi邻居
    stats=["mean", "std_dev"], # 统计量：均值和标准差
)

In [71]:
# 拟合
ssf.fit([structure])
# 特征化
features = ssf.featurize(structure)
# 获取特征标签
feature_labels = ssf.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,mean CN_VoronoiNN,std_dev CN_VoronoiNN
0,14.0,0.0


In [75]:
# 直接对dataframe进行特征化
df = df_raw.copy()
ssf.featurize_dataframe(df, "oxi_structure")
df

SiteStatsFingerprint:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ..."
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ..."
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...


## Symmetry

In [76]:
from matminer.featurizers.structure.symmetry import GlobalSymmetryFeatures, Dimensionality

### 1. GlobalSymmetryFeatures
用于确定结构的对称性特征，例如空间群号和晶体系统。

In [77]:
gsf = GlobalSymmetryFeatures(desired_features=["spacegroup_num", "crystal_system", "crystal_system_int"])
# 特征化
features = gsf.featurize(structure)
# 获取特征标签
feature_labels = gsf.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,spacegroup_num,crystal_system,crystal_system_int
0,225,cubic,1


In [78]:
# 直接对dataframe进行特征化
df = df_raw.copy()
gsf = GlobalSymmetryFeatures(desired_features=["spacegroup_num", "crystal_system", "crystal_system_int"])
df = gsf.featurize_dataframe(df, "oxi_structure")
df

GlobalSymmetryFeatures:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,spacegroup_num,crystal_system,crystal_system_int
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",225,cubic,1
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",167,trigonal,3
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,160,trigonal,3


### 2. Dimensionality
用于确定结构的维度。
维度定义如下：
- 1：线性原子链或孤立原子/无键
- 2：分层
- 3：3D 连接结构

In [79]:
d = Dimensionality()
# 特征化
features = d.featurize(structure)
# 获取特征标签
feature_labels = d.feature_labels()
# 创建包含特征的df
feat_df = pd.DataFrame([features], columns=feature_labels)
feat_df

,dimensionality
0,3


In [80]:
# 直接对dataframe进行特征化
df = df_raw.copy()
d = Dimensionality()
df = d.featurize_dataframe(df, "oxi_structure")
df

Dimensionality:   0%|          | 0/3 [00:00<?, ?it/s]

,material_id,formula_pretty,composition,symmetry,nsites,structure,oxi_structure,dimensionality
0,mp-22862,NaCl,"(Na, Cl)",crystal_system=<CrystalSystem.cubic: 'Cubic'> ...,2,"[[0. 0. 0.] Na, [2.281343 1.613153 3.951403] Cl]","[[0. 0. 0.] Na+, [2.281343 1.613153 3.951403] ...",3
1,mp-3953,CaCO3,"(Ca, C, O)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,10,"[[3.21199713 2.07688527 7.58523127] Ca, [0. 0....","[[3.21199713 2.07688527 7.58523127] Ca2+, [0. ...",3
2,mp-1204356,SiC,"(Si, C)",crystal_system=<CrystalSystem.trig: 'Trigonal'...,38,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,[[ 1.42104952e-21 -5.83008571e-05 -5.73687857e...,3
